In [17]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import plotly.express as px
import numpy as np

In [9]:
train_panel_in = "../../../data/final/train_model.parquet"
bike_panel_in = "../../../data/final/bike_model.parquet"
uber_panel_in = "../../../data/final/uber_model.parquet"

placebo_out = "../../../reports/replication/placebo.md"

In [10]:
panel = pd.concat([pd.read_parquet(train_panel_in),
                   pd.read_parquet(bike_panel_in),
                   pd.read_parquet(uber_panel_in)],
                  keys=['train','bike','uber'], names=['transit']).reset_index('transit')

# Placebo Test

In [11]:
def placebo_data(df, transit):
    df = df.loc[df.transit == transit]
    df = df.loc[df.date < "2024-08-19"]
    df = df.sort_values('date')
    df['time'] = pd.Categorical(df['date']).codes
    # Compute windows for placebo dates
    window_size = 4 # days
    n_windows = 31 // window_size
    df['windows'] = (df['time'].max() - df['time']) // window_size # count backwards
    pre = n_windows + 1 - df['windows'].clip(lower=0, upper=n_windows+1) # count forwards
    df['pre'] = pre.astype(str)
    # df['pre_label'] = (df['time'].max() - df.groupby('pre')['time'].transform('min')).astype(str)
    df = df.loc[df.groupby('pre')['UCMP'].transform('nunique') == 2] # some dont have data
    return df

def model_placebo(df, transit):
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    dist_terms = ['log_train_distance','log_bike_distance','log_bus_distance']
    dist_terms = [c for c in dist_terms if c in df.columns and transit not in c]
    dist_term = " + ".join(dist_terms)
    dist_term = dist_term if dist_term else "1"
    formula = f"""np.log1p(rides) ~ 
                UCMP * C(pre) +
                time + C(dotw) +
                {dist_term} +
                lat + long + I(lat * long) +
                I(lat**2) + I(long**2)
                """
    model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
    return model

In [12]:
model_data = placebo_data(panel, 'train')
train_pt_model = model_placebo(model_data, 'train')
model_data = placebo_data(panel, 'bike')
bike_pt_model = model_placebo(model_data, 'bike')
model_data = placebo_data(panel, 'uber')
uber_pt_model = model_placebo(model_data, 'uber')

In [13]:
constraints = [x for x in uber_pt_model.model.exog_names if 'UCMP:C(pre)' in x]
print(uber_pt_model.f_test(','.join([f"({c} = 0)" for c in constraints])))

constraints = [x for x in train_pt_model.model.exog_names if 'UCMP:C(pre)' in x]
print(train_pt_model.f_test(','.join([f"({c} = 0)" for c in constraints])))

constraints = [x for x in bike_pt_model.model.exog_names if 'UCMP:C(pre)' in x]
print(bike_pt_model.f_test(','.join([f"({c} = 0)" for c in constraints])))

<F test: F=3.3549288527905854, p=0.0008237628710338064, df_denom=1.23e+03, df_num=8>
<F test: F=5.946135988771264, p=1.4541062622251937e-06, df_denom=141, df_num=8>
<F test: F=3.2249039669600448, p=0.0012148088321174357, df_denom=1.49e+03, df_num=8>


In [14]:
regs = [x for x in bike_pt_model.model.exog_names if 'UCMP:C(pre)' in x] + \
    [x for x in train_pt_model.model.exog_names if 'UCMP:C(pre)' in x] + \
    [x for x in uber_pt_model.model.exog_names if 'UCMP:C(pre)' in x]
headers = ["PT (Uber)", "PT (Train)", "PT (Bike)"]
pcb_summary = summary_col([uber_pt_model, train_pt_model, bike_pt_model], 
            model_names=headers,
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=sorted(list(set(regs))),
            drop_omitted=True)

pre_to_date = model_data.groupby('pre')['date'].min() + ' to ' + model_data.groupby('pre')['date'].max()
pre_to_date = pre_to_date.to_dict()
pat = r'UCMP:C\(pre\)\[T.(\d)\]'
pre_dates = pcb_summary.tables[0].index.str.extract(pat).map(pre_to_date.get)
is_date = pcb_summary.tables[0].index.str.match(pat)
pcb_summary.tables[0].index = pcb_summary.tables[0].index.where(~is_date, pre_dates)

pcb_tbl = tabulate(pcb_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
with open(placebo_out,"w") as f:
    f.write(pcb_tbl)
print(pcb_tbl)


|                          | PT (Uber)   | PT (Train)   | PT (Bike)   |
|--------------------------|-------------|--------------|-------------|
| 2024-06-24 to 2024-06-27 | 0.0178      | 0.0107       | 0.1032**    |
|                          | (0.0638)    | (0.0328)     | (0.0451)    |
| 2024-07-01 to 2024-07-04 | -0.0691*    | -1.3482      | -0.1382**   |
|                          | (0.0383)    | (1.1665)     | (0.0575)    |
| 2024-07-08 to 2024-07-11 | 0.1224      | -1.3918      | -0.0778     |
|                          | (0.0828)    | (1.1920)     | (0.0479)    |
| 2024-07-15 to 2024-07-18 | -0.0840*    | -1.3755      | 0.0330      |
|                          | (0.0485)    | (1.1953)     | (0.0440)    |
| 2024-07-22 to 2024-07-25 | -0.0789*    | -1.3959      | 0.0575      |
|                          | (0.0472)    | (1.1998)     | (0.0505)    |
| 2024-07-29 to 2024-08-01 | 0.0878**    | -1.2910      | 0.0748      |
|                          | (0.0361)    | (1.0977)     | (0.047

In [36]:
main_effect = pd.concat([
    train_pt_model.params[train_pt_model.params.index.str.startswith('UCMP:C(pre)')],
    bike_pt_model.params[bike_pt_model.params.index.str.startswith('UCMP:C(pre)')],
    uber_pt_model.params[uber_pt_model.params.index.str.startswith('UCMP:C(pre)')]],
    keys=['train','bike','uber'], names=['transit']).reset_index('transit')
main_effect = main_effect.rename(columns={0:'coef'})
px.box(main_effect, x='transit',y='coef')